# Pretrained audio encoders

BEATs and CLAP: frozen embeddings + logistic regression, CLAP zero-shot, and BEATs fine-tuning
with or without velocity consistency. Windows are 0.5 s; predictions average their probabilities.

Run this notebook alone with Python 3.12+ and a GPU. All experiment code is here; packages, data
and pretrained models are downloaded automatically. The data archive supplies only two recording
tables and two waveform arrays; no repository Python files are imported.
`QUICK = True` uses one window per recording, one fold and 30 updates. Increase `ACCUMULATION_STEPS`
if memory runs out. Results go to `analysis/classification/pretrained/` (`pretrained_quick/` for checks).
Finished stages are reused; start a fresh results folder after changing settings.


In [ ]:
QUICK = False

DRIVE_FILE_ID = "1NmRCDqvGvmNwQGC53IzTu9jJRaI8Newk"

INPUTS = ["audio", "acceleration", "audio + acceleration"]
FINETUNE_INPUTS = ["audio", "audio + acceleration"]
CONSISTENCY_WEIGHTS = [0.0, 1.0]
HELD_OUT_VELOCITY = 60

N_UPDATES = 1500
PAIRS_PER_BATCH = 32
ACCUMULATION_STEPS = 2
ENCODER_LR = 2e-5
HEAD_LR = 1e-3
WAVE_RMS = 0.1

## Setup: packages, data and models

In [ ]:
import importlib.util
import subprocess
import sys

PACKAGES = {
    "numpy": "numpy",
    "torch": "torch",
    "torchaudio": "torchaudio",
    "sklearn": "scikit-learn",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "transformers": "transformers",
}
missing = [package for module, package in PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("installing", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

    import site

    site.addsitedir(site.getusersitepackages())
    importlib.invalidate_caches()

In [ ]:
import tarfile
import urllib.request
from pathlib import Path

ROOT = Path.cwd()
DATA_FILES = [
    "analysis/classification/waveforms.npz",
    "analysis/classification/sens3_waveforms.npz",
    "analysis/signal_processing/recordings.csv",
    "analysis/classification/sens3_recordings.csv",
]


def download_from_drive(file_id, path):

    url = f"https://drive.usercontent.google.com/download?id={file_id}&export=download&confirm=t"
    partial = path.with_suffix(".part")
    done = partial.stat().st_size if partial.exists() else 0
    request = urllib.request.Request(url, headers={"Range": f"bytes={done}-"} if done else {})
    with urllib.request.urlopen(request) as response, open(partial, "ab" if done else "wb") as out:
        if done and response.status != 206:
            out.truncate(0)
            done = 0
        assert "html" not in response.headers.get("Content-Type", ""), "Drive sent a web page: is the file shared?"
        total = done + int(response.headers.get("Content-Length", 0))
        while chunk := response.read(1 << 22):
            out.write(chunk)
            done += len(chunk)
            print(f"\r{done / 2**30:.2f} / {total / 2**30:.2f} GB", end="", flush=True)
    print()
    partial.rename(path)


if not all((ROOT / name).exists() for name in DATA_FILES):
    archive_path = ROOT / "data.tar"
    if not archive_path.exists():
        download_from_drive(DRIVE_FILE_ID, archive_path)
    with tarfile.open(archive_path) as archive:
        archive.extractall(ROOT, members=[archive.getmember(name) for name in DATA_FILES], filter="data")
    archive_path.unlink()
print("data ready in", ROOT)

MODELS = ROOT / "models"
BEATS_DIR = MODELS / "beats"
BEATS_DIR.mkdir(parents=True, exist_ok=True)
BEATS_CODE = "https://raw.githubusercontent.com/microsoft/unilm/master/beats/"

BEATS_WEIGHTS = "https://huggingface.co/alibabasglab/beats/resolve/main/BEATs_iter3_plus_AS2M.pt"
for name in ["BEATs.py", "backbone.py", "modules.py"]:
    if not (BEATS_DIR / name).exists():
        urllib.request.urlretrieve(BEATS_CODE + name, BEATS_DIR / name)
if not (BEATS_DIR / "BEATs_iter3_plus_AS2M.pt").exists():
    print("downloading BEATs weights (360 MB)...")
    urllib.request.urlretrieve(BEATS_WEIGHTS, BEATS_DIR / "BEATs_iter3_plus_AS2M.pt")
CLAP_NAME = "laion/clap-htsat-unfused"
print("BEATs ready in", BEATS_DIR)

In [ ]:
import numpy as np
import pandas as pd
import torch
import torchaudio
from torch import nn

sys.path.insert(0, str(BEATS_DIR))
from BEATs import BEATs, BEATsConfig  # noqa: E402

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    properties = torch.cuda.get_device_properties(0)
    print(f"GPU: {properties.name}, {properties.total_memory / 2**30:.0f} GB")
else:
    print("no GPU: the embeddings will be slow and the fine-tuning far too slow")

AMP_DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8 else torch.float16

## Evaluation

Five surface folds, stratified by family (seed 42), separately per dataset. Training weights
balance (setup, family) groups. Balanced accuracy with 95% surface-bootstrap intervals; chance 0.25.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

FAMILIES = ["wood", "metal", "textile", "polymer"]
N_FOLDS = 5
CHANCE = 1 / len(FAMILIES)
OUT = ROOT / "analysis/classification"


def surface_folds(meta):
    surfaces = meta[["surface", "family"]].drop_duplicates().reset_index(drop=True)
    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_of_surface = {}
    for fold, (_, test) in enumerate(splitter.split(surfaces.surface, surfaces.family)):
        for surface in surfaces.surface.iloc[test]:
            fold_of_surface[surface] = fold
    return meta.surface.map(fold_of_surface).to_numpy()


def folds(recordings):
    fold = pd.Series(0, index=recordings.index)
    for dataset in recordings.dataset.unique():
        in_dataset = recordings.dataset == dataset
        fold[in_dataset] = surface_folds(recordings[in_dataset])
    return fold.to_numpy()


def group_weights(recordings):
    weights = 1 / recordings.groupby(["dataset", "family"]).family.transform("size").to_numpy()
    return weights / weights.mean()


def fit_and_predict(model, train_features, train_families, weights, test_features):
    scaler = StandardScaler().fit(train_features)
    model.fit(scaler.transform(train_features), train_families, sample_weight=weights)
    return model.predict(scaler.transform(test_features))


def mean_recall(confusion):
    return np.mean(np.diag(confusion) / confusion.sum(axis=1))


def score(recordings, predicted):
    predicted = np.asarray(predicted)
    confusions = {}
    for surface in recordings.surface.unique():
        on_surface = (recordings.surface == surface).to_numpy()
        confusions[surface] = confusion_matrix(recordings.family[on_surface], predicted[on_surface], labels=FAMILIES)

    family_of_surface = recordings.groupby("surface").family.first()
    rng = np.random.default_rng(42)
    draws = []
    for _ in range(2000):
        resampled = np.zeros((len(FAMILIES), len(FAMILIES)))
        for family in FAMILIES:
            surfaces = family_of_surface.index[family_of_surface == family]
            for surface in rng.choice(surfaces, size=len(surfaces)):
                resampled += confusions[surface]
        draws.append(mean_recall(resampled))

    low, high = np.percentile(draws, [2.5, 97.5])
    return {"balanced accuracy": mean_recall(sum(confusions.values())), "low": low, "high": high}


def plot_scores(scores, groups, bars, panels, title, name):
    panel_names = scores[panels].unique()
    bar_names = scores[bars].unique()
    width = 0.8 / len(bar_names)

    fig, axes = plt.subplots(1, len(panel_names), figsize=(5 * len(panel_names), 4), sharey=True, squeeze=False)
    for ax, panel in zip(axes[0], panel_names, strict=True):
        in_panel = scores[scores[panels] == panel]
        group_names = in_panel[groups].unique()
        for k, bar in enumerate(bar_names):
            rows = in_panel[in_panel[bars] == bar].set_index(groups).reindex(group_names)
            x = np.arange(len(group_names)) + (k - (len(bar_names) - 1) / 2) * width
            below = rows["balanced accuracy"] - rows.low
            above = rows.high - rows["balanced accuracy"]
            ax.bar(x, rows["balanced accuracy"], width, color=f"C{k}", label=bar)
            ax.errorbar(x, rows["balanced accuracy"], yerr=[below, above], fmt="none", ecolor="black", capsize=2)
        ax.axhline(CHANCE, color="black", linestyle=":", linewidth=1)
        ax.set_xticks(range(len(group_names)), group_names, rotation=20)
        ax.set(title=panel, ylim=(0, 1))

    axes[0, 0].set_ylabel("balanced accuracy, 95% interval\n(dotted line: chance)")
    axes[0, -1].legend(title=bars, bbox_to_anchor=(1, 1), loc="upper left")
    fig.suptitle(title)
    fig.savefig(OUT / name, dpi=130, bbox_inches="tight")
    print("saved", OUT / name)


RESULTS = OUT / ("pretrained_quick" if QUICK else "pretrained")
RESULTS.mkdir(parents=True, exist_ok=True)

## Data

Unit-RMS windows: audio at 16 kHz, acceleration projected onto its dominant axis and resampled
from 8 to 16 kHz. Recording tables define the order of the waveform arrays.


In [ ]:
recordings = pd.read_csv(ROOT / "analysis/signal_processing/recordings.csv", dtype={"surface": str})
sens3 = pd.read_csv(OUT / "sens3_recordings.csv", dtype={"surface": str})
sens3["dataset"] = "sens3"


def load_waves(name):

    cache = np.load(OUT / name)
    recording_of_window = cache["recording"]
    keep = np.ones(len(recording_of_window), bool)
    if QUICK:
        keep[1:] = recording_of_window[1:] != recording_of_window[:-1]
    accel_8k = cache["accel"][keep]
    accel = torch.empty((len(accel_8k), 2 * accel_8k.shape[1]), dtype=torch.float16)
    for start in range(0, len(accel_8k), 4096):
        piece = torch.from_numpy(accel_8k[start : start + 4096].astype(np.float32))
        accel[start : start + 4096] = torchaudio.functional.resample(piece, 8000, 16000).half()
    waves = {"audio": torch.from_numpy(cache["audio"][keep]), "accel": accel}
    return waves, recording_of_window[keep]


waves, recording_of_window = load_waves("waveforms.npz")
sens3_waves, sens3_recording_of_window = load_waves("sens3_waveforms.npz")
SIGNALS_OF_INPUT = {"audio": ["audio"], "acceleration": ["accel"], "audio + acceleration": ["audio", "accel"]}

fold = folds(recordings)
in_cluster = (recordings.dataset == "cluster").to_numpy()
print(
    f"{len(recordings)} recordings, {len(recording_of_window)} windows; "
    f"SENS3: {len(sens3)} chunks, {len(sens3_recording_of_window)} windows"
)

## Frozen encoders

BEATs iter3+ (768 dimensions) and CLAP HTSAT (512). CLAP repeats each window to fill its 10 s input.


In [ ]:
def load_beats():
    checkpoint = torch.load(BEATS_DIR / "BEATs_iter3_plus_AS2M.pt", map_location="cpu", weights_only=True)
    model = BEATs(BEATsConfig(checkpoint["cfg"]))
    model.load_state_dict(checkpoint["model"])

    preprocess = model.preprocess

    def preprocess_in_float32(source, **kwargs):
        with torch.autocast("cuda", enabled=False):
            return preprocess(source.float(), **kwargs)

    model.preprocess = preprocess_in_float32
    return model.to(DEVICE)


def beats_embedding(model, x):

    features, _ = model.extract_features(x.float() * WAVE_RMS)
    return features.mean(dim=1)


@torch.no_grad()
def embed_all(embed, signal_waves, batch_size):
    out = []
    for start in range(0, len(signal_waves), batch_size):
        x = signal_waves[start : start + batch_size].to(DEVICE)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=DEVICE == "cuda"):
            out.append(embed(x).float().cpu())
    return torch.cat(out).numpy()


def cached_embeddings(encoder, make_embed, batch_size):
    path = RESULTS / f"embeddings_{encoder}.npz"
    if not path.exists():
        embed = make_embed()
        arrays = {}
        for prefix, all_waves in [("", waves), ("sens3_", sens3_waves)]:
            for signal_name, signal_waves in all_waves.items():
                arrays[prefix + signal_name] = embed_all(embed, signal_waves, batch_size)
                print(encoder, prefix + signal_name, arrays[prefix + signal_name].shape, flush=True)
        np.savez(path, **arrays)
        del embed
        torch.cuda.empty_cache()
    return dict(np.load(path))


def make_beats_embed():
    model = load_beats().eval()
    return lambda x: beats_embedding(model, x)


beats_embeddings = cached_embeddings("beats", make_beats_embed, batch_size=256)

In [ ]:
from transformers import ClapModel, ClapProcessor

clap_processor = ClapProcessor.from_pretrained(CLAP_NAME)


def features(output):

    return output if torch.is_tensor(output) else output.pooler_output


def clap_input_features(x48):

    extractor = clap_processor.feature_extractor
    fusion = extractor.truncation == "fusion"
    filters = torch.as_tensor(
        extractor.mel_filters if fusion else extractor.mel_filters_slaney, dtype=torch.float32, device=x48.device
    )
    window = torch.hann_window(extractor.fft_window_size, device=x48.device)
    n = extractor.nb_max_samples
    tiled = x48.repeat(1, n // x48.shape[1])
    tiled = nn.functional.pad(tiled, (0, n - tiled.shape[1]))
    power = (
        torch.stft(
            tiled,
            extractor.fft_window_size,
            extractor.hop_length,
            window=window,
            center=True,
            pad_mode="reflect",
            return_complex=True,
        ).abs()
        ** 2
    )
    mel = 10 * torch.log10(torch.clamp(filters.T @ power, min=1e-10)).transpose(1, 2)
    return mel[:, None].expand(-1, 4 if fusion else 1, -1, -1)


def make_clap_embed():
    model = ClapModel.from_pretrained(CLAP_NAME).to(DEVICE).eval()

    def embed(x):
        with torch.autocast("cuda", enabled=False):
            x48 = torchaudio.functional.resample(x.float(), 16000, 48000) * WAVE_RMS
            input_features = clap_input_features(x48)
        is_longer = torch.zeros(len(x), 1, dtype=torch.bool, device=x.device)
        return features(model.get_audio_features(input_features=input_features, is_longer=is_longer))

    return embed


clap_embeddings = cached_embeddings("clap", make_clap_embed, batch_size=64)

### Linear probes

Mean embedding per recording. Train-only standardization. Evaluate new Cluster surfaces,
Cluster → LMT108 and Cluster + LMT108 → SENS3.


In [ ]:
from sklearn.linear_model import LogisticRegression


def per_recording(window_embeddings, recording_of_window, n_recordings):
    return pd.DataFrame(window_embeddings).groupby(recording_of_window).mean().reindex(range(n_recordings)).to_numpy()


def recording_features(embeddings, input_name, prefix=""):
    rec_of_window, n = (sens3_recording_of_window, len(sens3)) if prefix else (recording_of_window, len(recordings))
    signals = [embeddings[prefix + name] for name in SIGNALS_OF_INPUT[input_name]]
    return per_recording(np.concatenate(signals, axis=1), rec_of_window, n)


def linear_probe_scores(encoder, embeddings):
    def fit(features, train, test_features):
        return fit_and_predict(
            LogisticRegression(max_iter=3000),
            features[train],
            recordings.family[train],
            group_weights(recordings[train]),
            test_features,
        )

    rows = []
    for input_name in INPUTS:
        X, X_sens3 = recording_features(embeddings, input_name), recording_features(embeddings, input_name, "sens3_")
        has_windows = ~np.isnan(X).any(axis=1)
        predicted = pd.Series(index=recordings.index, dtype=object)

        for k in range(1 if QUICK else N_FOLDS):
            train, test = in_cluster & (fold != k) & has_windows, in_cluster & (fold == k)
            predicted[test] = fit(X, train, X[test])
        predicted[~in_cluster] = fit(X, in_cluster, X[~in_cluster])
        sens3_predicted = fit(X, has_windows, X_sens3)

        tested_cluster = in_cluster & predicted.notna().to_numpy()
        for test_name, recs, pred in [
            ("Cluster, new surfaces", recordings[tested_cluster], predicted[tested_cluster]),
            ("LMT108, transfer", recordings[~in_cluster], predicted[~in_cluster]),
            ("SENS3, external", sens3, sens3_predicted),
        ]:
            rows.append(
                {
                    "input": input_name,
                    "model": f"{encoder} + logistic",
                    "test": test_name,
                    **score(recs.reset_index(drop=True), np.asarray(pred)),
                }
            )
    return pd.DataFrame(rows)


probe_scores = pd.concat([linear_probe_scores("BEATs", beats_embeddings), linear_probe_scores("CLAP", clap_embeddings)])
probe_scores.to_csv(RESULTS / "linear_probe_scores.csv", index=False)
probe_scores.round(3)

### CLAP zero-shot

Audio only: compare against four text prompts and average window probabilities per recording.


In [ ]:
PROMPTS = {family: f"the sound of a probe sliding on a {family} surface" for family in FAMILIES}
PROMPTS["polymer"] = "the sound of a probe sliding on a plastic surface"

clap_model = ClapModel.from_pretrained(CLAP_NAME).to(DEVICE).eval()
with torch.no_grad():
    tokens = clap_processor.tokenizer(list(PROMPTS.values()), padding=True, return_tensors="pt").to(DEVICE)
    text = features(clap_model.get_text_features(**tokens)).float().cpu().numpy()
del clap_model
torch.cuda.empty_cache()


def zero_shot(audio_embeddings, rec_of_window):
    a = audio_embeddings / np.linalg.norm(audio_embeddings, axis=1, keepdims=True)
    t = text / np.linalg.norm(text, axis=1, keepdims=True)
    probabilities = torch.from_numpy(100 * a @ t.T).softmax(dim=1).numpy()
    return pd.DataFrame(probabilities, columns=FAMILIES).groupby(rec_of_window).mean().idxmax(axis=1)


rows = []
for test_name, recs, emb, rec_of_window in [
    ("Cluster, new surfaces", recordings[in_cluster], clap_embeddings["audio"], recording_of_window),
    ("LMT108, transfer", recordings[~in_cluster], clap_embeddings["audio"], recording_of_window),
    ("SENS3, external", sens3, clap_embeddings["sens3_audio"], sens3_recording_of_window),
]:
    predicted = zero_shot(emb, rec_of_window).reindex(recs.index)
    rows.append(
        {
            "input": "audio",
            "model": "CLAP zero-shot",
            "test": test_name,
            **score(recs.reset_index(drop=True), predicted.to_numpy()),
        }
    )
zero_shot_scores = pd.DataFrame(rows)
zero_shot_scores.to_csv(RESULTS / "zero_shot_scores.csv", index=False)
zero_shot_scores.round(3)

## Fine-tuning and velocity consistency

Shared BEATs encoder, dropout and linear head. Each batch contains Cluster pairs (same surface,
direction, force and repeat, different velocities) plus random training windows. Loss: weighted
cross-entropy + symmetric KL on pairs. Both penalty settings use the same batches and seeds.

1. New Cluster surfaces: five folds, all velocities.
2. New surfaces at 60 mm/s: exclude that velocity from training.
3. New setup: Cluster → LMT108, then Cluster + LMT108 → SENS3.

SENS3 has already been inspected in earlier experiments; these results remain exploratory.


In [ ]:
class FineTunedBEATs(nn.Module):
    def __init__(self, signal_names):
        super().__init__()
        self.signal_names = signal_names
        self.encoder = load_beats()
        width = self.encoder.cfg.encoder_embed_dim * len(signal_names)
        self.head = nn.Sequential(nn.Dropout(0.1), nn.Linear(width, len(FAMILIES)))

    def forward(self, batch):
        return self.head(torch.cat([beats_embedding(self.encoder, batch[name]) for name in self.signal_names], dim=1))


first_window = np.searchsorted(recording_of_window, np.arange(len(recordings)))
n_windows = np.bincount(recording_of_window, minlength=len(recordings))
family_index = recordings.family.map({name: k for k, name in enumerate(FAMILIES)}).to_numpy()


def random_windows(recs, rng):
    return first_window[recs] + rng.integers(0, n_windows[recs])


def velocity_pairs(train):

    keys = ["surface", "direction", "force", "repeat"]
    cluster = recordings[train & in_cluster].reset_index()
    merged = cluster.merge(cluster, on=keys, suffixes=("_a", "_b"))
    merged = merged[merged.velocity_a < merged.velocity_b]
    return merged[["index_a", "index_b"]].to_numpy()


def batch_of(windows, signal_names, signal_waves=None):
    signal_waves = waves if signal_waves is None else signal_waves
    return {name: signal_waves[name][windows].to(DEVICE, non_blocking=True) for name in signal_names}


def symmetric_kl(logits_a, logits_b):
    log_a, log_b = logits_a.log_softmax(dim=1), logits_b.log_softmax(dim=1)
    return 0.5 * ((log_a.exp() * (log_a - log_b)).sum(1) + (log_b.exp() * (log_b - log_a)).sum(1)).mean()


def finetune(input_name, train, weight, seed):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = FineTunedBEATs(SIGNALS_OF_INPUT[input_name]).to(DEVICE)
    optimizer = torch.optim.AdamW(
        [{"params": model.encoder.parameters(), "lr": ENCODER_LR}, {"params": model.head.parameters(), "lr": HEAD_LR}],
        weight_decay=0.01,
    )
    n_updates = 30 if QUICK else N_UPDATES
    schedule = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=[ENCODER_LR, HEAD_LR], total_steps=n_updates, pct_start=0.1
    )
    scaler = torch.amp.GradScaler(enabled=torch.float16 == AMP_DTYPE)

    pairs = velocity_pairs(train)
    train_recs = np.flatnonzero(train)
    recording_weight = np.zeros(len(recordings))
    recording_weight[train] = group_weights(recordings[train])

    model.train()
    for _ in range(n_updates):
        chosen = pairs[rng.integers(0, len(pairs), PAIRS_PER_BATCH)]
        extra = rng.choice(train_recs, PAIRS_PER_BATCH)
        recs = np.concatenate([chosen[:, 0], chosen[:, 1], extra])
        windows = random_windows(recs, rng)
        total_weight = recording_weight[recs].sum()

        optimizer.zero_grad()

        for piece in np.array_split(np.arange(PAIRS_PER_BATCH), ACCUMULATION_STEPS):
            rows = np.concatenate([piece, PAIRS_PER_BATCH + piece, 2 * PAIRS_PER_BATCH + piece])
            targets = torch.as_tensor(family_index[recs[rows]], device=DEVICE)
            weights = torch.as_tensor(recording_weight[recs[rows]], device=DEVICE, dtype=torch.float32)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=DEVICE == "cuda"):
                logits = model(batch_of(windows[rows], model.signal_names)).float()
            losses = nn.functional.cross_entropy(logits, targets, reduction="none")
            loss = (losses * weights).sum() / total_weight
            if weight > 0:
                n = len(piece)
                loss = loss + weight * symmetric_kl(logits[:n], logits[n : 2 * n]) * n / PAIRS_PER_BATCH
            scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        schedule.step()
    return model


@torch.no_grad()
def classify(model, test_recs, rec_of_window, signal_waves):

    model.eval()
    windows = np.flatnonzero(np.isin(rec_of_window, test_recs))
    probabilities = []
    for start in range(0, len(windows), 256):
        batch = batch_of(windows[start : start + 256], model.signal_names, signal_waves)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=DEVICE == "cuda"):
            probabilities.append(model(batch).float().softmax(dim=1).cpu())
    probabilities = pd.DataFrame(torch.cat(probabilities).numpy(), columns=FAMILIES)
    return probabilities.groupby(rec_of_window[windows]).mean().idxmax(axis=1)

In [ ]:
import time

PREDICTIONS = RESULTS / "finetune_predictions.csv"
seen_velocity = (recordings.velocity != HELD_OUT_VELOCITY).to_numpy()
everything = np.ones(len(recordings), bool)


TRAININGS = []
for k in range(1 if QUICK else N_FOLDS):
    held_out = in_cluster & (fold == k)
    TRAININGS.append(("new surfaces", k, in_cluster & (fold != k), [("Cluster, new surfaces", held_out, "main")]))
    TRAININGS.append(
        (
            "unseen velocity",
            k,
            in_cluster & (fold != k) & seen_velocity,
            [("Cluster, new surfaces at unseen velocity", held_out & ~seen_velocity, "main")],
        )
    )
TRAININGS.append(("new setup", 0, in_cluster, [("LMT108, transfer", ~in_cluster, "main")]))
TRAININGS.append(("new setup", 1, everything, [("SENS3, external", np.ones(len(sens3), bool), "sens3")]))

done = set()
if PREDICTIONS.exists():
    previous = pd.read_csv(PREDICTIONS)
    done = set(previous[["protocol", "fold", "input", "weight"]].drop_duplicates().itertuples(index=False, name=None))

for input_name in FINETUNE_INPUTS:
    for weight in CONSISTENCY_WEIGHTS:
        for protocol, k, train, tests in TRAININGS:
            if (protocol, k, input_name, weight) in done:
                continue
            started = time.time()
            model = finetune(input_name, train, weight, seed=k)
            rows = []
            for test_name, tested, dataset in tests:
                rec_of_window, signal_waves = (
                    (sens3_recording_of_window, sens3_waves) if dataset == "sens3" else (recording_of_window, waves)
                )
                predicted = classify(model, np.flatnonzero(tested), rec_of_window, signal_waves)
                rows += [
                    {
                        "protocol": protocol,
                        "fold": k,
                        "input": input_name,
                        "weight": weight,
                        "test": test_name,
                        "dataset": dataset,
                        "recording": r,
                        "predicted": p,
                    }
                    for r, p in predicted.items()
                ]
            pd.DataFrame(rows).to_csv(PREDICTIONS, mode="a", header=not PREDICTIONS.exists(), index=False)
            del model
            torch.cuda.empty_cache()
            print(f"{protocol}, fold {k}, {input_name}, weight {weight}: {time.time() - started:.0f} s", flush=True)

## Scores and figures

In [ ]:
predictions = pd.read_csv(PREDICTIONS)
rows = []
for (test_name, input_name, weight, dataset), group in predictions.groupby(["test", "input", "weight", "dataset"]):
    recs = (sens3 if dataset == "sens3" else recordings).loc[group.recording].reset_index(drop=True)
    model_name = "BEATs fine-tuned" + (f" + consistency {weight:g}" if weight > 0 else "")
    rows.append(
        {"input": input_name, "model": model_name, "test": test_name, **score(recs, group.predicted.to_numpy())}
    )
finetune_scores = pd.DataFrame(rows)
finetune_scores.to_csv(RESULTS / "finetune_scores.csv", index=False)

all_scores = pd.concat([probe_scores, zero_shot_scores, finetune_scores], ignore_index=True)
all_scores.to_csv(RESULTS / "pretrained_scores.csv", index=False)
all_scores.pivot_table(index=["test", "input"], columns="model", values="balanced accuracy").round(2)

In [ ]:
plot_scores(
    all_scores[all_scores.test != "Cluster, new surfaces at unseen velocity"],
    groups="input",
    bars="model",
    panels="test",
    title="Pretrained audio encoders",
    name=f"{RESULTS.name}/pretrained_encoders.png",
)
plot_scores(
    finetune_scores,
    groups="input",
    bars="model",
    panels="test",
    title="Fine-tuned BEATs with and without the velocity-consistency penalty",
    name=f"{RESULTS.name}/consistency_penalty.png",
)
plt.show()